# Análise Final — Interrupções de Carga do ONS

## Objetivo

Responder às seis perguntas de negócio formuladas para o MVP, utilizando as tabelas da camada Gold e o esquema estrela implementado no Databricks.

A análise contempla o período de 2007 a 2026, sendo 2026 um ano parcial, com dados até 01/09/2026.

Os indicadores são calculados a partir dos 8.309 registros válidos, após a remoção de 16 duplicatas integrais e o isolamento de um registro com valores negativos de tempo e energia não suprida.

### Premissas de interpretação

- Registro: linha individual da tabela fato.
- Perturbação: ocorrência identificada pelo código original da fonte.
- ENS: energia não suprida, expressa em MWh.
- Carga interrompida: magnitude registrada em MW.
- Tempo de recomposição: duração média informada em cada registro, expressa em minutos.

A quantidade de registros não equivale necessariamente à quantidade de perturbações, pois uma perturbação pode envolver diferentes agentes e localidades.

As somas de carga interrompida representam magnitudes acumuladas dos registros, não potência simultânea do sistema.

As análises temporais utilizam a data efetiva registrada, e não o ano incorporado ao código da perturbação.

Os resultados descrevem os registros disponíveis na fonte e não constituem indicadores normalizados de desempenho de distribuidoras ou subsistemas.


In [0]:
%sql

SELECT
    cod_perturbacao,
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT din_interrupcaocarga) AS qtd_datas_distintas,
    MIN(din_interrupcaocarga) AS primeira_data,
    MAX(din_interrupcaocarga) AS ultima_data
FROM ons_energia.gold.fato_interrupcao
GROUP BY cod_perturbacao
HAVING COUNT(DISTINCT din_interrupcaocarga) > 1
ORDER BY qtd_datas_distintas DESC, qtd_registros DESC;



## P1 — Como evoluíram a frequência das interrupções e a energia não suprida ao longo dos anos?

### Objetivo

Investigar a evolução temporal das interrupções de carga, identificando períodos com maior frequência de registros e maior impacto energético.

Serão avaliados:

- Quantidade anual de registros;
- Quantidade de códigos de perturbação distintos;
- Energia não suprida acumulada;
- Soma das cargas interrompidas registradas.

A análise busca verificar se o aumento da frequência está necessariamente associado ao aumento da severidade energética.


In [0]:
%sql

SELECT
    t.ano,
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT f.cod_perturbacao)
        AS qtd_codigos_perturbacao,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh,
    ROUND(SUM(f.val_cargainterrompida_mw), 2)
        AS soma_carga_mw,
    CASE
        WHEN t.ano = 2026
            THEN 'Parcial até 01/09/2026'
        ELSE 'Ano completo na cobertura da base'
    END AS cobertura
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_tempo t
    ON f.sk_tempo = t.sk_tempo
GROUP BY t.ano
ORDER BY t.ano;


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.



### Interpretação — Evolução anual

Os resultados demonstram que a frequência dos registros e a energia não suprida apresentam comportamentos distintos ao longo do histórico.

O ano de 2023 apresentou a maior quantidade de registros, com 573 interrupções registradas e 440 códigos de perturbação distintos. Nesse período, a ENS acumulada foi de 63.958,83 MWh.

Entretanto, o maior impacto energético anual ocorreu em 2009, com 107.825,41 MWh, apesar de apresentar 474 registros.

O ano de 2020 também apresentou impacto expressivo, com 98.919,49 MWh distribuídos em 318 registros.

Essa diferença evidencia que o impacto energético anual pode ser fortemente influenciado por ocorrências individuais ou perturbações de grande abrangência, mesmo em anos com frequência relativamente menor.

O ano de 2026 possui apenas 196 registros e 13.789,78 MWh até 01/09/2026. Por se tratar de um período parcial, seus totais não devem ser comparados diretamente com os anos completos.


In [0]:
%sql

SELECT
    t.ano,
    COUNT(*) AS qtd_registros,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_tempo t
    ON f.sk_tempo = t.sk_tempo
WHERE t.ano BETWEEN 2007 AND 2025
GROUP BY t.ano
ORDER BY t.ano;



### Interpretação — Frequência versus impacto energético

A comparação gráfica entre quantidade de registros e energia não suprida demonstra que as duas medidas não evoluem de maneira proporcional.

Em 2009 e 2020, observam-se picos expressivos de ENS sem que esses anos apresentem a maior frequência de registros.

Em 2023, ocorre o comportamento inverso: a maior frequência anual não corresponde ao maior volume de energia não suprida.

Portanto, a avaliação histórica exige indicadores complementares de frequência e severidade. A utilização isolada da quantidade de registros pode ocultar períodos de elevado impacto energético.




## P2 — Quais estados e subsistemas concentram os maiores volumes de carga interrompida e energia não suprida?

### Objetivo

Investigar a distribuição geográfica dos impactos registrados, identificando diferenças entre unidades federativas e subsistemas elétricos.

A análise considera:

- Frequência de registros;
- Soma das cargas interrompidas;
- Energia não suprida acumulada.

As agregações por estado e subsistema serão realizadas separadamente, preservando seus respectivos níveis geográficos.


In [0]:
%sql

SELECT
    l.id_estado,
    COUNT(*) AS qtd_registros,
    ROUND(SUM(f.val_cargainterrompida_mw), 2)
        AS soma_carga_mw,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade
GROUP BY l.id_estado
ORDER BY ens_mwh DESC;


Databricks visualization. Run in Databricks to view.


### Interpretação — Distribuição por estado

São Paulo apresentou o maior volume acumulado de energia não suprida, com 117.406,88 MWh em 549 registros. Também apresentou a maior soma de carga interrompida, com 98.000,14 MW.

O Amapá apresentou o segundo maior volume de ENS, com 88.038,08 MWh, apesar de possuir apenas 108 registros.

Esse resultado é fortemente influenciado pela perturbação 6814/2020, cujo registro individual apresenta 77.509,63 MWh de energia não suprida.

O Pará apresentou 77.349,68 MWh em 847 registros, enquanto o Rio de Janeiro acumulou 77.089,15 MWh em 276 registros.

O Rio Grande do Sul apresentou a maior frequência de registros entre os estados, com 1.020, mas sua ENS acumulada foi de 44.545,46 MWh.

Esses resultados demonstram que estados com maior quantidade de registros não necessariamente apresentam maior impacto energético acumulado.

As diferenças devem ser interpretadas como características dos registros disponíveis, sem representar diretamente uma comparação normalizada de desempenho entre estados.


In [0]:
%sql

SELECT
    l.id_subsistema,
    l.nom_subsistema,
    COUNT(*) AS qtd_registros,
    ROUND(SUM(f.val_cargainterrompida_mw), 2)
        AS soma_carga_mw,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade
GROUP BY
    l.id_subsistema,
    l.nom_subsistema
ORDER BY ens_mwh DESC;


Databricks visualization. Run in Databricks to view.



### Interpretação — Distribuição por subsistema

O subsistema Sudeste/Centro-Oeste apresentou o maior volume acumulado de energia não suprida, com 315.060,07 MWh, distribuídos em 3.316 registros.

O Norte apresentou 294.339,02 MWh em 2.233 registros, aproximando-se do volume energético do Sudeste/Centro-Oeste, embora apresente menor frequência.

O Nordeste acumulou 203.725,38 MWh em 1.318 registros, enquanto o Sul apresentou 83.991,71 MWh em 1.442 registros.

A comparação reforça que a frequência não explica isoladamente a magnitude energética das interrupções.

Esses valores são absolutos e não foram normalizados pela carga atendida, extensão da rede, quantidade de consumidores ou outras características estruturais dos subsistemas.




## P3 — Como varia o tempo médio de recomposição entre estados e subsistemas?

### Objetivo

Comparar os tempos de recomposição registrados nas diferentes localidades e identificar a influência de ocorrências de longa duração.

Serão calculados:

- Tempo médio de recomposição;
- Tempo mediano de recomposição;
- Maior tempo individual registrado;
- Quantidade de registros analisados.

A mediana será utilizada como medida complementar à média, pois a avaliação de qualidade identificou valores extremos que podem elevar significativamente a média aritmética.

O indicador representa a média dos tempos informados por registro, sem ponderação pela carga interrompida.


In [0]:
%sql

SELECT
    l.id_estado,
    COUNT(*) AS qtd_registros,
    ROUND(AVG(f.val_tempomedio_minutos), 2)
        AS tempo_medio_min,
    ROUND(
        PERCENTILE_APPROX(f.val_tempomedio_minutos, 0.5),
        2
    ) AS tempo_mediano_min,
    ROUND(MAX(f.val_tempomedio_minutos), 2)
        AS maior_tempo_min
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade
GROUP BY l.id_estado
ORDER BY tempo_medio_min DESC;


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.


### Interpretação — Tempo de recomposição por estado

O Amapá apresentou tempo médio de 378,73 minutos e mediana de 40,75 minutos. A diferença entre essas medidas evidencia a influência de ocorrências de longa duração, especialmente o registro com 29.203 minutos.

Roraima apresentou média de 139,47 minutos e mediana de apenas 10 minutos, também demonstrando elevada assimetria.

O Rio Grande do Sul apresentou média de 125,81 minutos e mediana de 13,43 minutos. O estado contém o maior tempo individual da base, de 68.102,70 minutos, equivalente a aproximadamente 47,3 dias.

A Paraíba apresentou média de 72,69 minutos e mediana de 29 minutos em 88 registros.

No extremo inferior da distribuição das médias, Mato Grosso do Sul apresentou 18,45 minutos, com mediana de 4 minutos.

As diferenças entre média e mediana indicam que ocorrências excepcionais exercem influência relevante sobre os indicadores agregados. Por isso, a interpretação do tempo de recomposição não deve se limitar à média aritmética.

Os resultados não representam, isoladamente, uma avaliação de desempenho operacional dos estados, pois não consideram diferenças estruturais ou exposição ao risco.


In [0]:
%sql

SELECT
    l.nom_subsistema,
    COUNT(*) AS qtd_registros,
    ROUND(AVG(f.val_tempomedio_minutos), 2)
        AS tempo_medio_min,
    ROUND(
        PERCENTILE_APPROX(f.val_tempomedio_minutos, 0.5),
        2
    ) AS tempo_mediano_min,
    ROUND(MAX(f.val_tempomedio_minutos), 2)
        AS maior_tempo_min
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade
GROUP BY l.nom_subsistema
ORDER BY tempo_medio_min DESC;


Databricks visualization. Run in Databricks to view.


### Interpretação — Tempo de recomposição por subsistema

O subsistema Sul apresentou o maior tempo médio de recomposição, com 103,19 minutos, mas sua mediana foi de apenas 12 minutos.

O Norte apresentou média de 89,64 minutos e mediana de 11,90 minutos.

O Nordeste apresentou média de 53,90 minutos e mediana de 22,95 minutos, enquanto o Sudeste/Centro-Oeste apresentou média de 40,33 minutos e mediana de 14 minutos.

A diferença entre as duas medidas é especialmente expressiva no Sul e no Norte, indicando forte influência dos registros extremos.

Embora o Sul apresente a maior média, o Nordeste apresenta a maior mediana entre os quatro subsistemas.

Portanto, a conclusão depende da medida estatística utilizada: a média evidencia a influência dos eventos de longa duração, enquanto a mediana descreve o valor central da distribuição dos registros.



## P4 — Como diferem as interrupções que envolveram ou não a Rede Básica?

### Objetivo

Comparar os registros classificados pelo ONS conforme o envolvimento da Rede Básica, avaliando diferenças de frequência, carga interrompida, tempo de recomposição e energia não suprida.

A comparação será descritiva, sem atribuir causalidade ao envolvimento da Rede Básica.

Serão analisados indicadores totais e médios para evitar que diferenças na quantidade de registros sejam confundidas com diferenças na magnitude individual das interrupções.


In [0]:
%sql

SELECT
    r.envolve_rede_basica,
    COUNT(*) AS qtd_registros,
    ROUND(SUM(f.val_cargainterrompida_mw), 2)
        AS soma_carga_mw,
    ROUND(AVG(f.val_cargainterrompida_mw), 2)
        AS carga_media_mw,
    ROUND(AVG(f.val_tempomedio_minutos), 2)
        AS tempo_medio_min,
    ROUND(
        PERCENTILE_APPROX(f.val_tempomedio_minutos, 0.5),
        2
    ) AS tempo_mediano_min,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_total_mwh,
    ROUND(AVG(f.val_energianaosuprida_mwh), 2)
        AS ens_media_mwh
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_rede r
    ON f.sk_rede = r.sk_rede
GROUP BY r.envolve_rede_basica
ORDER BY r.envolve_rede_basica;


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.


### Interpretação — Envolvimento da Rede Básica

Os registros classificados como envolvendo a Rede Básica totalizaram 6.210 ocorrências, enquanto os classificados como não envolvendo totalizaram 2.099.

No grupo com envolvimento da Rede Básica, a carga interrompida média foi de 95,94 MW por registro, superior aos 55,24 MW observados no grupo sem envolvimento.

A energia não suprida acumulada também foi maior no primeiro grupo: 755.980,96 MWh, contra 141.135,21 MWh.

A ENS média por registro foi de 121,74 MWh nas interrupções com envolvimento da Rede Básica e 67,24 MWh nas demais.

O comportamento do tempo de recomposição apresentou uma particularidade: a média foi maior no grupo sem envolvimento da Rede Básica, com 79,19 minutos, em comparação com 62,40 minutos no grupo com envolvimento.

Entretanto, a mediana apresentou comportamento inverso: 16 minutos para os registros com envolvimento da Rede Básica e 9,38 minutos para os demais.

Essa diferença evidencia a influência dos valores extremos sobre a média e reforça a necessidade de considerar múltiplas medidas estatísticas.

Os resultados demonstram associação entre o indicador de envolvimento da Rede Básica e diferenças nas magnitudes registradas, mas não permitem estabelecer relação causal.



## P5 — Existe comportamento mensal ou sazonal na frequência e magnitude das interrupções?

### Objetivo

Investigar a distribuição das interrupções ao longo dos meses, identificando períodos de maior frequência e energia não suprida.

A análise considera os anos de 2007 a 2025, excluindo 2026 por sua cobertura parcial.

Serão avaliados:

- Quantidade acumulada de registros por mês;
- Média anual de registros por mês;
- Energia não suprida acumulada;
- Média anual da ENS por mês;
- Carga interrompida média por registro.

A distribuição mensal será complementada por uma análise ano a ano para distinguir padrões recorrentes de concentrações provocadas por perturbações excepcionais.


In [0]:
%sql

SELECT
    t.mes,
    COUNT(*) AS qtd_registros,
    ROUND(COUNT(*) / 19.0, 2)
        AS media_registros_por_ano,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_total_mwh,
    ROUND(
        SUM(f.val_energianaosuprida_mwh) / 19.0,
        2
    ) AS ens_media_anual_mwh,
    ROUND(AVG(f.val_cargainterrompida_mw), 2)
        AS carga_media_mw
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_tempo t
    ON f.sk_tempo = t.sk_tempo
WHERE t.ano BETWEEN 2007 AND 2025
GROUP BY t.mes
ORDER BY t.mes;


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.


### Interpretação — Distribuição mensal

Outubro apresentou a maior frequência acumulada, com 940 registros, correspondentes a uma média de 49,47 registros por ano.

Março e setembro apresentaram 789 registros cada, enquanto julho apresentou a menor frequência acumulada, com 425 registros e média de 22,37 por ano.

Em relação à energia não suprida, novembro apresentou o maior volume acumulado, com 202.462,71 MWh, seguido por outubro, com 127.955,03 MWh, e março, com 121.649,24 MWh.

Agosto também apresentou impacto expressivo, com 107.037,83 MWh, além da maior carga interrompida média por registro, de 111,52 MW.

Os resultados indicam concentração histórica de registros em determinados meses, especialmente outubro.

Entretanto, os totais mensais de ENS podem ser fortemente influenciados por eventos excepcionais. Portanto, a distribuição agregada não é suficiente para comprovar um padrão sazonal recorrente de severidade.


In [0]:
%sql

SELECT
    t.ano,
    t.mes,
    COUNT(*) AS qtd_registros,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_tempo t
    ON f.sk_tempo = t.sk_tempo
WHERE t.ano BETWEEN 2007 AND 2025
GROUP BY t.ano, t.mes
ORDER BY t.ano, t.mes;


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.


### Interpretação — Recorrência e influência de eventos excepcionais

A análise ano a ano demonstra que parte dos picos mensais de energia não suprida está associada a períodos específicos.

Em novembro de 2009, foram registrados 91.369,37 MWh, enquanto novembro de 2020 apresentou 78.778,19 MWh. Esses dois períodos exercem influência relevante sobre o elevado total histórico de novembro.

Em março de 2018, a ENS alcançou 52.787,15 MWh. Em agosto de 2023, foram registrados 49.145,39 MWh, acompanhados de 150 registros no mês.

Outubro de 2012 também apresentou concentração expressiva, com 44.489,70 MWh.

Esses resultados demonstram que a concentração mensal de ENS não é uniforme ao longo dos anos.

Assim, é possível identificar diferenças no perfil mensal dos registros, mas não atribuir automaticamente os picos de energia não suprida a uma sazonalidade recorrente.

A identificação das causas exigiria informações complementares sobre os eventos, condições meteorológicas e características operacionais.



## P6 — Quais perturbações concentraram os maiores volumes de energia não suprida?

### Objetivo

Identificar os códigos de perturbação associados aos maiores impactos energéticos do histórico, investigando a concentração da ENS e a abrangência dos eventos.

A análise considera:

- Quantidade de registros associados ao código;
- Quantidade de estados envolvidos;
- Quantidade de agentes distintos;
- Energia não suprida acumulada;
- Soma das cargas interrompidas registradas.

Uma perturbação pode envolver diferentes agentes, estados e parcelas de carga interrompida. Por esse motivo, os registros serão consolidados pelo código original, sem selecionar arbitrariamente uma única linha.

Os resultados representam a agregação das medidas registradas na fonte, após o tratamento de qualidade.


In [0]:
%sql

SELECT
    f.cod_perturbacao,
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT l.id_estado) AS qtd_estados,
    COUNT(DISTINCT a.nom_agente) AS qtd_agentes,
    ROUND(SUM(f.val_energianaosuprida_mwh), 2)
        AS ens_mwh,
    ROUND(SUM(f.val_cargainterrompida_mw), 2)
        AS soma_carga_mw
FROM ons_energia.gold.fato_interrupcao f
JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade
JOIN ons_energia.gold.dim_agente a
    ON f.sk_agente = a.sk_agente
GROUP BY f.cod_perturbacao
ORDER BY ens_mwh DESC
LIMIT 20;


### Interpretação — Concentração da energia não suprida

A perturbação 0672/2011 apresentou o maior volume acumulado de energia não suprida, com 89.371,48 MWh, distribuídos em 29 registros, 18 estados e 29 agentes.

Embora o código contenha o ano de 2011, a data registrada para essa perturbação é de 2009. Portanto, seu impacto integra os indicadores temporais de 2009.

A perturbação 6814/2020 apresentou o segundo maior volume, com 77.509,63 MWh concentrados em apenas um registro, referente ao Amapá.

Esse caso demonstra que uma ocorrência individual de longa duração pode apresentar impacto energético superior ao de perturbações distribuídas entre diversos agentes.

A perturbação 5799/2018 acumulou 51.692,18 MWh em 66 registros, abrangendo 26 estados e 62 agentes.

A perturbação 5598/2023 acumulou 48.106,22 MWh em 116 registros, envolvendo 26 estados e 99 agentes distintos.

A perturbação 5783/2012 também apresentou impacto relevante, com 37.518,45 MWh em 14 registros.

Os resultados evidenciam dois comportamentos distintos: perturbações de ampla abrangência, com impactos distribuídos entre diversos agentes e localidades, e ocorrências individuais com elevada energia não suprida.

Consequentemente, a quantidade de registros ou de estados envolvidos não deve ser utilizada isoladamente como medida de severidade energética.



In [0]:
%sql


WITH ens_por_perturbacao AS (
    SELECT
        cod_perturbacao,
        COUNT(*) AS qtd_registros,
        SUM(val_energianaosuprida_mwh) AS ens_mwh
    FROM ons_energia.gold.fato_interrupcao
    GROUP BY cod_perturbacao
),
participacao AS (
    SELECT
        cod_perturbacao,
        qtd_registros,
        ens_mwh,
        100.0 * ens_mwh / SUM(ens_mwh) OVER ()
            AS participacao_percentual
    FROM ens_por_perturbacao
)
SELECT
    cod_perturbacao,
    qtd_registros,
    ROUND(ens_mwh, 2) AS ens_mwh,
    ROUND(participacao_percentual, 2)
        AS participacao_percentual
FROM participacao
ORDER BY ens_mwh DESC
LIMIT 20;


### Interpretação — Participação no impacto energético total

A energia não suprida apresenta concentração expressiva em um conjunto reduzido de códigos de perturbação. A perturbação `0672/2011` responde por 9,96% da ENS da base e a `6814/2020`, por 8,64%.

Os cinco códigos com maior ENS concentram aproximadamente **33,90%** do total; os dez primeiros, **45,17%**; e os vinte primeiros, **53,06%**. Esses percentuais são aproximados e refletem os resultados arredondados da consulta.

A concentração ocorre tanto em perturbações de ampla abrangência, com registros de diversos agentes e estados, quanto em ocorrências individuais de longa duração. A consulta de granularidade deste notebook não retornou códigos associados a mais de uma data de interrupção na versão analisada, permitindo o agrupamento por código nesta análise.

Os percentuais representam os registros válidos da fonte utilizada, e não a totalidade independente de impactos no sistema elétrico brasileiro.


# Conclusões da análise

As seis perguntas de negócio demonstraram que frequência, duração, abrangência geográfica e energia não suprida são dimensões complementares na avaliação das interrupções de carga.

A evolução histórica apresentou oscilações expressivas, com períodos de elevada energia não suprida que não coincidiram necessariamente com os anos de maior frequência.

A distribuição geográfica mostrou que estados e subsistemas com menor quantidade de registros podem apresentar impactos energéticos acumulados elevados.

A análise dos tempos de recomposição revelou distribuições assimétricas, nas quais ocorrências extremas influenciam significativamente as médias. A utilização conjunta de média e mediana permitiu uma interpretação mais completa.

Os registros envolvendo a Rede Básica apresentaram maiores valores médios de carga interrompida e energia não suprida, enquanto o comportamento dos tempos de recomposição variou conforme a medida estatística utilizada.

A análise mensal identificou diferenças na frequência histórica, mas também demonstrou que os totais de energia não suprida podem ser fortemente influenciados por eventos excepcionais.

Por fim, a consolidação por código de perturbação evidenciou a concentração de impactos em eventos de diferentes características, desde ocorrências individuais de longa duração até perturbações de ampla abrangência geográfica.

## Limitações

A base representa os registros de interrupção de carga disponibilizados pelo ONS, não a totalidade das interrupções de distribuição.

O ano de 2026 possui cobertura parcial até 01/09/2026.

Foram identificados 201 registros com divergência entre o ano do código da perturbação e o ano da data registrada. As análises temporais utilizaram a data informada pela fonte.

Os valores extremos positivos foram preservados, pois sua magnitude isolada não comprova erro.

As comparações geográficas apresentam indicadores absolutos, sem normalização por carga atendida, número de consumidores ou características da rede.

Os resultados devem ser interpretados como análises descritivas dos dados disponíveis, sem atribuição automática de causalidade.

